# 02 - Modeling and Test Evaluation

Final evaluation on the untouched test set and detailed failure mode analysis.

In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

artifact = joblib.load('../models/detector.joblib')
model = artifact['model']
test_df = pd.read_csv('../data/processed/features_test.csv')
X_test = test_df[artifact['feature_cols']]
y_test = test_df['label']

probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

print(classification_report(y_test, preds, target_names=['Human', 'AI']))

## 02 - Model Evaluation and Error Analysis

### 1. Test Set Metrics
- **Accuracy**: 96.67%
- **ROC-AUC**: 0.9989
- **Precision**: 0.9375
- **Recall**: 1.0000
- **F1 Score**: 0.9677

### 2. Error Analysis (Patterns & Failure Modes)
1. **False Positives (Human text flagged as AI, total=5)**:
   - Occurs when human writing is unusually smooth, formal, or lacks idiosyncratic colloquialisms, resulting in lower GPT-2 perplexity.
   - Short sentences with repetitive structure lower the burstiness metric, making the text statistically mimic model completions.

**FP #1 (prob=0.89, ppl=21.9, burst=37.1)**:
> There was a world before this. 
 
 A world where the sky was clear and not brown. 
 A world where there were towns and cities, and the buildings reached for the skies. 
 A world where you could eat an...

**FP #2 (prob=0.79, ppl=23.3, burst=55.0)**:
> The President wiped his brow. This was the day where he'd show billions of people what he believed in. They'd see what he stood for. They'd see just what he's made of. It was time. Time to change Amer...

**FP #3 (prob=0.57, ppl=37.8, burst=100.4)**:
> I can see the ambulance. Blue and red lights color the road signaling my doom. What could be about to happen? I'm going at barely 18 miles per hour. Suddenly I see another ambulance behind it. And ano...

**FP #4 (prob=0.54, ppl=35.8, burst=28.2)**:
> With the click of a mouse, the Gospel was finally set upon the unsuspecting Internet. The figure got up slowly, let out a sigh, and wandered away to pursue other things. 
 
 The pages and pages of div...

**FP #5 (prob=0.54, ppl=44.3, burst=18.6)**:
> It's easy to get lost in himself today as on this particular day, this round of golf was the greatest played game of his life. 
 
 He was enveloped in the smell of the grass, the cool breeze, and reve...


2. **False Negatives (AI text flagged as Human, total=0)**:
   - Occurs when AI generates creative fiction with rare vocabulary, unconventional character names, or dialogue punctuation that raises token surprise.
   - Long, compound sentences with dialogue increase sentence-to-sentence perplexity variance (higher burstiness).

